In [2]:
import json
import os
import re
import pandas as pd
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
client = Groq(api_key=os.environ['GROQ_API_KEY'])

with open('../data/adverse_action_shap_data.json') as f:
    adverse_action_data = json.load(f)

print(f"Loaded {len(adverse_action_data)} rejected applicant records")
print(adverse_action_data[0])

Loaded 125 rejected applicant records
{'applicant_index': 0, 'top_factors': [{'feature': 'other_installment_plans_none', 'shap_value': 0.32545483112335205, 'feature_value': 0.0}, {'feature': 'employment_since_ 1 year', 'shap_value': 0.21354694664478302, 'feature_value': 1.0}, {'feature': 'checking_account_no checking account', 'shap_value': 0.21089528501033783, 'feature_value': 0.0}, {'feature': 'credit_history_critical account/other credits', 'shap_value': 0.08728121221065521, 'feature_value': 0.0}, {'feature': 'duration', 'shap_value': 0.06526990979909897, 'feature_value': 18.0}]}


In [3]:
# (description if the dummy = 1, description if the dummy = 0)
# Keys are the cleaned column names ('<' and '>' removed for XGBoost)
TRANSLATIONS = {
    'other_installment_plans_none': ('you do not have other installment plans with another lender', 'you have other installment plans with another lender or store'),
    'other_installment_plans_stores': ('you have installment plans with a store', 'you do not have installment plans with a store'),
    'checking_account_no checking account': ('you do not have a checking account', 'you have a checking account'),
    'checking_account_ 0 DM': ('your checking account balance is below 0 DM', 'your checking account balance is not below 0 DM'),
    'checking_account_= 200 DM': ('your checking account balance is 200 DM or more', 'your checking account balance is below 200 DM'),
    'credit_history_critical account/other credits': ('you have a critical credit history or other existing credits at other banks', 'you do not have a critical credit history'),
    'credit_history_delay in past': ('you have had delays in paying off past credits', 'you have not had payment delays'),
    'credit_history_existing credits paid duly': ('your existing credits have been paid on time', 'your existing credits have not all been paid on time'),
    'credit_history_no credits/all paid': ('you have no credits or all credits are fully paid', 'you have outstanding, unpaid credits'),
    'savings_account_ 100 DM': ('your savings account balance is below 100 DM', 'your savings account balance is 100 DM or more'),
    'savings_account_500-1000 DM': ('your savings account balance is between 500 and 1000 DM', 'your savings balance is outside the 500-1000 DM range'),
    'savings_account_= 1000 DM': ('your savings account balance is 1000 DM or more', 'your savings balance is below 1000 DM'),
    'savings_account_unknown/no savings': ('you have no savings account or it is unknown', 'you have a known savings account'),
    'employment_since_ 1 year': ('you have been employed for less than 1 year', 'you have been employed for 1 year or more'),
    'employment_since_4-7 years': ('you have been employed for 4-7 years', 'your employment duration is outside 4-7 years'),
    'employment_since_= 7 years': ('you have been employed for 7 years or more', 'you have been employed for less than 7 years'),
    'employment_since_unemployed': ('you are currently unemployed', 'you are currently employed'),
    'property_unknown/no property': ('you have no property or it is unknown', 'you have known property'),
    'property_real estate': ('your property is real estate', 'you do not own real estate'),
    'property_car or other': ('your property is a car or other asset', 'your property is not a car or other asset'),
    'other_debtors_guarantor': ('you have a guarantor', 'you do not have a guarantor'),
    'other_debtors_none': ('you have no co-applicant or guarantor', 'you have a co-applicant or guarantor'),
    'purpose_car (new)': ('the loan purpose is a new car', 'the loan purpose is not a new car'),
    'purpose_car (used)': ('the loan purpose is a used car', 'the loan purpose is not a used car'),
    'purpose_education': ('the loan purpose is education', 'the loan purpose is not education'),
    'purpose_furniture/equipment': ('the loan purpose is furniture or equipment', 'the loan purpose is not furniture or equipment'),
    'purpose_radio/TV': ('the loan purpose is a radio or TV', 'the loan purpose is not a radio or TV'),
    'purpose_repairs': ('the loan purpose is repairs', 'the loan purpose is not repairs'),
    'purpose_retraining': ('the loan purpose is retraining', 'the loan purpose is not retraining'),
    'purpose_domestic appliances': ('the loan purpose is domestic appliances', 'the loan purpose is not domestic appliances'),
    'purpose_others': ('the loan purpose is classified as other', 'the loan purpose is not classified as other'),
    'housing_own': ('you own your housing', 'you do not own your housing'),
    'housing_rent': ('you rent your housing', 'you do not rent your housing'),
    'job_skilled employee/official': ('you are a skilled employee or official', 'you are not a skilled employee or official'),
    'job_unskilled resident': ('you are an unskilled resident worker', 'you are not an unskilled resident worker'),
    'job_unemployed/unskilled non-resident': ('you are unemployed or an unskilled non-resident', 'you do not fall in this employment category'),
    'telephone_yes, registered': ('you have a registered telephone', 'you do not have a registered telephone'),
}

def translate_feature(feature_name, feature_value):
    numeric = {
        'duration': f'the requested loan duration is {feature_value:.0f} months',
        'credit_amount': f'the requested credit amount is {feature_value:.0f} DM',
        'installment_rate': f'the installment rate is {feature_value:.0f} (on a 1-4 scale of income percentage)',
        'residence_since': f'you have lived at your current residence for {feature_value:.0f} years',
        'existing_credits': f'you currently have {feature_value:.0f} existing credit(s) at this bank',
        'num_dependents': f'you have {feature_value:.0f} dependent(s)',
    }
    if feature_name in numeric:
        return numeric[feature_name]
    if feature_name in TRANSLATIONS:
        true_desc, false_desc = TRANSLATIONS[feature_name]
        return true_desc if feature_value == 1.0 else false_desc
    return f"UNMAPPED:{feature_name} (value: {feature_value})"

for factor in adverse_action_data[0]['top_factors']:
    print(translate_feature(factor['feature'], factor['feature_value']))

you have other installment plans with another lender or store
you have been employed for less than 1 year
you have a checking account
you do not have a critical credit history
the requested loan duration is 18 months


In [4]:
def build_adverse_action_prompt(applicant_index, top_factors):
    reasons_text = "\n".join([
        f"- {translate_feature(f['feature'], f['feature_value'])} (relative contribution: {f['shap_value']:.3f})"
        for f in top_factors
    ])

    return f"""You are helping a lending institution generate an adverse action notice, which is a legally required explanation sent to an applicant whose loan application was denied, consistent with the U.S. Equal Credit Opportunity Act (ECOA).

You are given the top factors that a credit risk model identified as contributing to this applicant's denial. Your task is to write a short, professional, plain-language explanation (3-5 sentences) that:
1. States clearly that the application was not approved.
2. Explains the specific reasons in plain, respectful, non-technical language, based ONLY on the factors provided below. Do not invent, assume, or add any reason not explicitly listed.
3. Does not use technical terms like "SHAP," "model," "feature," or "algorithm" - write as a human loan officer would.
4. Avoids blaming or judgmental language - state facts neutrally.
5. Ends with a brief, factual note that the applicant has the right to request further clarification and a copy of the information used in the decision, consistent with ECOA requirements.

Factors contributing to this decision (ranked by importance, most significant first):
{reasons_text}

Write the adverse action explanation now:"""

example = adverse_action_data[0]
print(build_adverse_action_prompt(example['applicant_index'], example['top_factors']))

You are helping a lending institution generate an adverse action notice, which is a legally required explanation sent to an applicant whose loan application was denied, consistent with the U.S. Equal Credit Opportunity Act (ECOA).

You are given the top factors that a credit risk model identified as contributing to this applicant's denial. Your task is to write a short, professional, plain-language explanation (3-5 sentences) that:
1. States clearly that the application was not approved.
2. Explains the specific reasons in plain, respectful, non-technical language, based ONLY on the factors provided below. Do not invent, assume, or add any reason not explicitly listed.
3. Does not use technical terms like "SHAP," "model," "feature," or "algorithm" - write as a human loan officer would.
4. Avoids blaming or judgmental language - state facts neutrally.
5. Ends with a brief, factual note that the applicant has the right to request further clarification and a copy of the information used i

In [5]:
FEATURE_KEYWORDS = {
    'other_installment_plans_none': ['installment', 'plan', 'lender', 'store'],
    'other_installment_plans_stores': ['installment', 'store'],
    'checking_account_no checking account': ['checking'],
    'checking_account_ 0 DM': ['checking', 'below zero', 'negative'],
    'checking_account_= 200 DM': ['checking'],
    'employment_since_ 1 year': ['employ'],
    'employment_since_4-7 years': ['employ'],
    'employment_since_= 7 years': ['employ'],
    'employment_since_unemployed': ['unemploy', 'employ'],
    'credit_history_critical account/other credits': ['credit history', 'limited credit', 'credit'],
    'credit_history_delay in past': ['delay', 'credit history'],
    'credit_history_existing credits paid duly': ['credit history', 'paid'],
    'credit_history_no credits/all paid': ['credit history', 'paid'],
    'duration': ['month', 'duration', 'term', 'loan'],
    'credit_amount': ['amount', 'requested', 'dm', 'exceed'],
    'savings_account_ 100 DM': ['savings'],
    'savings_account_500-1000 DM': ['savings'],
    'savings_account_= 1000 DM': ['savings'],
    'savings_account_unknown/no savings': ['savings'],
    'existing_credits': ['credit account', 'existing credit', 'other credit', 'credit(s)'],
    'installment_rate': ['installment rate', 'income'],
    'residence_since': ['residence', 'resided', 'living'],
    'num_dependents': ['dependent'],
    'property_real estate': ['real estate', 'property'],
    'property_car or other': ['property', 'asset', 'car'],
    'property_unknown/no property': ['property', 'collateral'],
    'other_debtors_guarantor': ['guarantor'],
    'other_debtors_none': ['guarantor', 'co-applicant', 'co applicant'],
    'purpose_car (new)': ['car', 'vehicle'],
    'purpose_car (used)': ['car', 'vehicle', 'used'],
    'purpose_education': ['education'],
    'purpose_furniture/equipment': ['furniture', 'equipment'],
    'purpose_radio/TV': ['radio', 'tv', 'television'],
    'purpose_repairs': ['repair'],
    'purpose_retraining': ['retrain'],
    'purpose_domestic appliances': ['appliance'],
    'purpose_others': ['purpose'],
    'housing_own': ['own', 'housing', 'homeowner'],
    'housing_rent': ['rent', 'housing'],
    'job_skilled employee/official': ['skilled', 'job', 'employment type', 'occupation'],
    'job_unskilled resident': ['unskilled', 'job', 'occupation'],
    'job_unemployed/unskilled non-resident': ['unemployed', 'unskilled', 'non-resident'],
    'telephone_yes, registered': ['telephone', 'phone'],
}

def normalize_text(text):
    # LLM output sometimes contains non-breaking hyphens (e.g. "real‑estate")
    return re.sub(r'[\u2010\u2011\u2012\u2013\u2014-]', ' ', text).lower()

def default_keywords(feature_name):
    return [p.lower() for p in feature_name.replace('_', ' ').split() if len(p) > 2]

def validate_notice(notice_text, top_factors):
    notice_norm = normalize_text(notice_text)
    rows = []
    for f in top_factors:
        keywords = FEATURE_KEYWORDS.get(f['feature'], default_keywords(f['feature']))
        rows.append({'feature': f['feature'],
                     'mentioned': any(normalize_text(k) in notice_norm for k in keywords)})
    return pd.DataFrame(rows)

In [6]:
def generate_adverse_action_notice(prompt, model="openai/gpt-oss-120b"):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=800,
        reasoning_effort="low",
    )
    return response.choices[0].message.content

prompt = build_adverse_action_prompt(example['applicant_index'], example['top_factors'])
live_notice = generate_adverse_action_notice(prompt)

print(live_notice)
print("\n--- validation ---")
print(validate_notice(live_notice, example['top_factors']))

We regret to inform you that your loan application was not approved. The decision was based on several factors, including the presence of other installment payment obligations, having been employed for less than one year, and the fact that you currently have a checking account but lack a longer‑standing credit history. Additionally, the 18‑month loan term you requested was taken into consideration. Under the Equal Credit Opportunity Act, you have the right to request a further explanation of this decision and to obtain a copy of the information used in evaluating your application.

--- validation ---
                                         feature  mentioned
0                   other_installment_plans_none       True
1                       employment_since_ 1 year       True
2           checking_account_no checking account       True
3  credit_history_critical account/other credits       True
4                                       duration       True


In [7]:
with open('../data/adverse_action_notices_final.json') as f:
    saved_notices = json.load(f)

missing = [r['applicant_index'] for r in saved_notices if not r['notice']]
failed = []
for r in saved_notices:
    if r['notice']:
        v = validate_notice(r['notice'], r['top_factors'])
        if not v['mentioned'].all():
            failed.append(r['applicant_index'])

print(f"Saved notices: {len(saved_notices)}")
print(f"Missing notices: {len(missing)}")
print(f"Fully validated: {len(saved_notices) - len(failed) - len(missing)} / {len(saved_notices)}")
print("Failed applicants:", failed)

print("\nSample saved notice (applicant", saved_notices[0]['applicant_index'], "):")
print(saved_notices[0]['notice'])

Saved notices: 125
Missing notices: 0
Fully validated: 125 / 125
Failed applicants: []

Sample saved notice (applicant 0 ):
We regret to inform you that your loan application was not approved. The decision was based on several factors, including the presence of other installment plans you currently have, having been employed for less than one year, and the fact that you maintain a checking account rather than a longer‑standing credit history. Additionally, the requested loan term of 18 months and the limited length of your credit history were also considered. Under the Equal Credit Opportunity Act, you may request a further explanation of this decision and a copy of the information used in evaluating your application.


In [8]:
PROTECTED_PREFIXES = ('personal_status_sex', 'foreign_worker', 'age')

# 1. No protected attribute should ever reach the LLM as a stated reason
protected_hits = [(r['applicant_index'], f['feature'])
                  for r in saved_notices for f in r['top_factors']
                  if f['feature'].startswith(PROTECTED_PREFIXES)]
print("Protected attributes found in factor lists:", len(protected_hits))

# 2. No factor should fall through to the raw-name fallback in the translator
unmapped = {f['feature'] for r in saved_notices for f in r['top_factors']
            if translate_feature(f['feature'], f['feature_value']).startswith('UNMAPPED')}
print("Features with no plain-English mapping:", unmapped)

# 3. Every factor should push toward rejection
non_positive = [(r['applicant_index'], f['feature']) for r in saved_notices
                for f in r['top_factors'] if f['shap_value'] <= 0]
print("Factors with non-positive SHAP value:", len(non_positive))

# 4. Quick scan of the notice text for protected-class wording (review any hits by hand)
protected_terms = re.compile(r'\b(sex|gender|male|female|marital|married|single|foreign|national origin|age|aged)\b', re.I)
text_hits = [r['applicant_index'] for r in saved_notices if r['notice'] and protected_terms.search(r['notice'])]
print("Notices containing protected-class wording:", len(text_hits), text_hits[:10])

Protected attributes found in factor lists: 0
Features with no plain-English mapping: set()
Factors with non-positive SHAP value: 0
Notices containing protected-class wording: 0 []
